## Helpful file for extracting low and high difference pairs from each narrative,KPI pair.
### Changle claim20 and agent number in path for extracting pairs from a specific combination

In [ ]:
from itertools import combinations
from collections import Counter
import os
import pandas as pd

# Change the claim and the agent number to extract pairs from a different
# narrative and KPI combination.
CLAIM = "claim20"
AGENT = "CN_Creator_Agent_3"
SOURCE = os.path.join("..", "refinement_per_claim_final_results", CLAIM, f"{AGENT}.csv")

# Create a list to hold the pairwise comparisons
df = pd.read_csv(SOURCE)

# Create a list to hold the pairwise comparisons
pairwise_data = []
for claim, group in df.groupby('claim'):
    cn_list = group[['counter_narrative', 'avg_share']].reset_index(drop=True)
    for (i, row1), (j, row2) in combinations(cn_list.iterrows(), 2):
        cn_1, score_1 = row1['counter_narrative'], row1['avg_share']
        cn_2, score_2 = row2['counter_narrative'], row2['avg_share']
        pairwise_data.append({
            'claim': claim,
            'cn_1': cn_1,
            'cn_2': cn_2,
            'avg_score_cn_1': score_1,
            'avg_score_cn_2': score_2,
            'diff': abs(score_1 - score_2)
        })

# Convert to DataFrame
pairs_df = pd.DataFrame(pairwise_data)

# Compute quantile thresholds
low_thresh = pairs_df['diff'].quantile(0.2)
high_thresh = pairs_df['diff'].quantile(0.5)

# Get relatively small and largest difference subsets
small_diff_df = pairs_df[pairs_df['diff'] <= low_thresh].sort_values(by='diff', ascending=False)
large_diff_df = pairs_df[pairs_df['diff'] >= high_thresh].sort_values(by='diff', ascending=False)

# Helper function to select pairs with CN repetition constraints
def select_pairs(source_df, max_pairs, used_counter, existing_pairs):
    selected = []
    for _, row in source_df.iterrows():
        if len(selected) >= max_pairs:
            break
        cn_1, cn_2 = row['cn_1'], row['cn_2']
        if used_counter[cn_1] < 2 and used_counter[cn_2] < 2:
            selected.append(row)
            used_counter[cn_1] += 1
            used_counter[cn_2] += 1
    return existing_pairs + selected

# Select 3 from each group, respecting repetition limit
used_counter = Counter()
selected_rows = []

selected_rows = select_pairs(large_diff_df, 3, used_counter, selected_rows)
selected_rows = select_pairs(small_diff_df, 3, used_counter, selected_rows)

# Final result
final_df = pd.DataFrame(selected_rows).drop(columns='diff')
final_df

In [ ]:
final_df.to_csv('temp_comparisons_data.csv', mode='a', header=False, index=False, encoding='utf-8-sig')